<a href="https://colab.research.google.com/github/ldhenriquez/analysis_players_football__01/blob/main/Taller_01_Pandas_Jugadores_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Taller Pandas - Estadísticas de Jugadores 2024/2025

Dataset: Football Players Stats 2024-2025 (Kaggle)

**Preguntas:**
1. ¿Cuál es el promedio de goles y asistencias según la posición?
2. ¿Qué liga tiene los jugadores más agresivos (tarjetas por 90 min)?
3. ¿Cuáles son los 10 jugadores más eficientes en goles por 90 min (con al menos 900 minutos jugados)?

In [3]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv('players_data-2024_2025.csv')
df.shape

(2268, 267)

In [5]:
df.head(3)

,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,...,Att (GK),Thr,Launch%,AvgLen,Opp,Stp,Stp%,#OPA,#OPA/90,AvgDist
0,1,Max Aarons,eng ENG,DF,Bournemouth,eng Premier League,24.0,2000.0,3,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Max Aarons,eng ENG,"DF,MF",Valencia,es La Liga,24.0,2000.0,4,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Rodrigo Abajas,es ESP,DF,Valencia,es La Liga,21.0,2003.0,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# son muchas columnas, me quedo solo con las que voy a usar
cols = ['Player', 'Nation', 'Pos', 'Squad', 'Comp', 'Age', 'MP', 'Min', '90s', 'Gls', 'Ast', 'G+A', 'CrdY', 'CrdR', 'xG', 'xAG']
df = df[cols]
df.head()

,Player,Nation,Pos,Squad,Comp,Age,MP,Min,90s,Gls,Ast,G+A,CrdY,CrdR,xG,xAG
0,Max Aarons,eng ENG,DF,Bournemouth,eng Premier League,24.0,3,86,1.0,0,0,0,0,0,0.0,0.0
1,Max Aarons,eng ENG,"DF,MF",Valencia,es La Liga,24.0,4,120,1.3,0,0,0,2,0,0.0,0.0
2,Rodrigo Abajas,es ESP,DF,Valencia,es La Liga,21.0,1,65,0.7,0,0,0,1,0,0.1,0.0
3,James Abankwah,ie IRL,"DF,MF",Udinese,it Serie A,20.0,6,88,1.0,0,0,0,1,0,0.1,0.0
4,Keyliane Abdallah,fr FRA,FW,Marseille,fr Ligue 1,18.0,1,3,0.0,0,0,0,0,0,0.0,0.0


In [7]:
df.isnull().sum()

,0
Player,0
Nation,7
Pos,0
Squad,0
Comp,0
Age,8
MP,0
Min,0
90s,0
Gls,0


## Limpieza

In [8]:
df = df.dropna(subset=['Nation', 'Age'])
df['Age'] = df['Age'].astype(int)
df.shape

(2258, 16)

In [9]:
# algunos jugadores tienen posicion doble tipo 'FW,MF', me quedo con la principal
df['Pos'].value_counts().head(10)

,count
Pos,
DF,684
MF,470
FW,303
"FW,MF",246
"MF,FW",196
GK,156
"DF,MF",82
"MF,DF",62
"DF,FW",39


In [10]:
df['Pos_Principal'] = df['Pos'].apply(lambda x: x.split(',')[0])
df['Pos_Principal'].value_counts()

,count
Pos_Principal,
DF,805
MF,728
FW,569
GK,156


## Nuevas variables

In [11]:
df['Gls_por_90'] = np.where(df['90s'] > 0, (df['Gls'] / df['90s']).round(2), 0)
df['Total_Tarjetas'] = df['CrdY'] + df['CrdR']
df['Tarjetas_por_90'] = np.where(df['90s'] > 0, (df['Total_Tarjetas'] / df['90s']).round(3), 0)

df[['Player', 'Gls', 'Gls_por_90', 'Total_Tarjetas', 'Tarjetas_por_90']].head()

,Player,Gls,Gls_por_90,Total_Tarjetas,Tarjetas_por_90
0,Max Aarons,0,0.0,0,0.000
1,Max Aarons,0,0.0,2,1.538
2,Rodrigo Abajas,0,0.0,1,1.429
3,James Abankwah,0,0.0,1,1.000
4,Keyliane Abdallah,0,0.0,0,0.000


---
## Pregunta 1 - Promedio de goles y asistencias por posicion

In [12]:
df.groupby('Pos_Principal')[['Gls', 'Ast', 'G+A']].mean().round(2).sort_values('G+A', ascending=False)

,Gls,Ast,G+A
Pos_Principal,,,
FW,3.74,1.66,5.40
MF,1.53,1.42,2.95
DF,0.67,0.84,1.51
GK,0.00,0.10,0.10


In [13]:
# cuantos jugadores hay por posicion
df.groupby('Pos_Principal')['Player'].count()

,Player
Pos_Principal,
DF,805
FW,569
GK,156
MF,728


**Conclusion:** Los delanteros (FW) tienen los promedios mas altos tanto en goles como asistencias, lo cual tiene todo el sentido. Los mediocampistas tambien aportan bastante. Porteros y defensas casi no aparecen en estadisticas ofensivas.

---
## Pregunta 2 - Que liga es la mas agresiva?

In [14]:
# filtro jugadores con al menos 5 partidos para no distorsionar
df_5mp = df[df['MP'] >= 5].copy()
print(f'Jugadores con 5+ partidos: {len(df_5mp)}')

Jugadores con 5+ partidos: 1886


In [15]:
liga_stats = df_5mp.groupby('Comp').agg(
    Prom_CrdY=('CrdY', 'mean'),
    Prom_CrdR=('CrdR', 'mean'),
    Tarjetas_por_90=('Tarjetas_por_90', 'mean'),
    N_jugadores=('Player', 'count')
).round(3).sort_values('Tarjetas_por_90', ascending=False)

liga_stats

,Prom_CrdY,Prom_CrdR,Tarjetas_por_90,N_jugadores
Comp,,,,
es La Liga,3.404,0.161,0.241,411
de Bundesliga,2.548,0.118,0.235,323
it Serie A,2.818,0.142,0.227,424
eng Premier League,3.274,0.114,0.223,368
fr Ligue 1,2.622,0.156,0.201,360


In [16]:
print(f"Liga mas agresiva: {liga_stats['Tarjetas_por_90'].idxmax()}")

Liga mas agresiva: es La Liga


**Conclusion:** La Liga española lidera en agresividad con el mayor promedio de tarjetas por 90 min. La Ligue 1 resulta ser la mas disciplinada de las 5 grandes ligas, algo que no era obvio.

---
## Pregunta 3 - Top 10 goleadores mas eficientes (min. 900 min)

In [17]:
# solo delanteros y medios con minutos suficientes
df_ofensivos = df[(df['Min'] >= 900) & (df['Pos_Principal'].isin(['FW', 'MF']))].copy()
print(f'Jugadores en el filtro: {len(df_ofensivos)}')

Jugadores en el filtro: 660


In [18]:
top10 = df_ofensivos.sort_values('Gls_por_90', ascending=False).head(10)
top10[['Player', 'Squad', 'Comp', 'Age', 'Min', 'Gls', '90s', 'Gls_por_90']]

,Player,Squad,Comp,Age,Min,Gls,90s,Gls_por_90
697,Ousmane Dembélé,Paris S-G,fr Ligue 1,27,1730,21,19.2,1.09
1317,Harry Kane,Bayern Munich,de Bundesliga,31,2381,26,26.5,0.98
1691,Kylian Mbappé,Real Madrid,es La Liga,25,2907,31,32.3,0.96
330,Mika Biereth,Monaco,fr Ligue 1,21,1228,13,13.6,0.96
2201,Mateo Retegui,Atalanta,it Serie A,25,2383,25,26.5,0.94
1631,Omar Marmoush,Eint Frankfurt,de Bundesliga,25,1448,15,16.1,0.93
1483,Robert Lewandowski,Barcelona,es La Liga,35,2667,27,29.6,0.91
2088,Ayoze Pérez,Villarreal,es La Liga,31,1970,19,21.9,0.87
1047,Amine Gouiri,Marseille,fr Ligue 1,24,1048,10,11.6,0.86
2171,Gonçalo Ramos,Paris S-G,fr Ligue 1,23,1066,10,11.8,0.85


In [19]:
prom_top10 = top10['Gls_por_90'].mean().round(3)
prom_general = df_ofensivos['Gls_por_90'].mean().round(3)
print(f'Top 10 promedio: {prom_top10} | General: {prom_general} | Ratio: {round(prom_top10/prom_general, 1)}x')

Top 10 promedio: 0.935 | General: 0.208 | Ratio: 4.5x


**Conclusion:** El Top 10 es casi 5 veces mas eficiente que el promedio. Sorloth lidera con 1.15 goles por 90 min, por encima incluso de Mbappe y Lewandowski. Tiene sentido filtrar por minutos minimos, sin ese filtro aparecen jugadores que jugaron 10 minutos y metieron un gol, lo cual no es representativo.